# 🧵 AGL-YOLO: An Attention-Guided, Ghost-Lightweight YOLO Framework with Multi-Scale Feature Fusion for Real-Time Fabric Defect Detection

**Real-dataset training & evaluation notebook**

---

### ⚠️ What changed from the demo version
The original notebook trained on a **synthetic fabric generator** (fake textures with a random shape drawn on top) and printed a **hard-coded benchmark table**. Nothing in that version was a measured result.

This version:
1. Loads a **real YOLO-format fabric-defect dataset** (images + `.txt` labels + `data.yaml`), such as the Aliyun/Tianchi cloth-flaw dataset or a Roboflow fabric-defect dataset.
2. Uses a **real anchor-free target assignment + loss** (SIoU regression + focal objectness + BCE classification), replacing the placeholder `mean(abs(output))` loss.
3. Computes **real mAP@0.5 / mAP@0.5:0.95, precision, recall** on a held-out validation split using `torchmetrics`.
4. Measures **real FPS and GFLOPs** on your actual hardware, instead of printing invented numbers.
5. Optionally trains a real **YOLOv8n baseline** (via `ultralytics`) on the *same* data split so the comparison table is apples-to-apples.

### ⚠️ What you still need to do
- **Run this on a GPU** (Google Colab: Runtime → Change runtime type → GPU). CPU will run but training will be very slow.
- **Provide a real dataset.** Section 2 gives you two ready options (Roboflow download, or upload your own YOLO-format zip). This notebook cannot download files on your behalf without your credentials.
- **Expect modest accuracy** unless you train for many epochs (50–300) on a few thousand images — a handful of epochs on a small dataset will not match published paper numbers, and you should say so honestly in any report.
---

## ⚙️ Step 0: Environment Setup & Dependencies
Installs everything needed for real training/eval: PyTorch, `torchmetrics` (real mAP), `thop` (real FLOPs), `pyyaml` (data.yaml parsing), and optionally `ultralytics` (for the YOLOv8n baseline comparison) and `roboflow` (for one-line dataset download).

In [ ]:
# ====================================================================
# ⚙️ Step 0: Environment Setup & Dependencies
# ====================================================================
!pip -q install torchmetrics thop pyyaml tqdm roboflow ultralytics 2>/dev/null

import os, math, time, random, glob, shutil, zipfile
import numpy as np
import cv2
import yaml
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchmetrics.detection.mean_ap import MeanAveragePrecision

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[AGL-YOLO Engine] Running on Compute Device: {device}")
print(f"[PyTorch Version] {torch.__version__}")
if device.type != 'cuda':
    print("[WARNING] No GPU detected. Training will be slow. In Colab: Runtime > Change runtime type > GPU.")

## 📥 Section 1: Get a Real Fabric Defect Dataset
Pick **one** of the options below. Either way you need a folder that ends up looking like this (standard YOLO layout):

```
dataset/
├── data.yaml            # names: [class0, class1, ...], nc: N, train:/val: paths
├── train/
│   ├── images/*.jpg
│   └── labels/*.txt     # one line per box: class_id cx cy w h (normalized 0-1)
└── valid/
    ├── images/*.jpg
    └── labels/*.txt
```

**Option A — Roboflow (easiest, already YOLO-formatted).**
Get a free API key at roboflow.com, then use the Roboflow "Fabric defect detection" project (or any fabric-defect project you find on Roboflow Universe) and paste your key below.

**Option B — Your own dataset (e.g. Tianchi/Guangdong cloth-flaw data converted to YOLO format, ZJU-Leaper, TILDA, FabricSpotDefect, etc).**
Zip it in the layout above, upload it in Colab's file browser, and set `DATASET_ZIP_PATH` below.

Only run the cell for the option you're using.

In [ ]:
# ====================================================================
# 📥 Option A: Download a fabric-defect dataset from Roboflow
# ====================================================================
USE_ROBOFLOW = False   # <-- set True to use this option

if USE_ROBOFLOW:
    from roboflow import Roboflow
    ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"          # <-- paste your key
    WORKSPACE = "pulak-deb-roy"                      # example public workspace
    PROJECT   = "fabric-defect-detection-erdgo"      # example public project
    VERSION   = 12

    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(WORKSPACE).project(PROJECT)
    dataset = project.version(VERSION).download("yolov8")
    DATASET_ROOT = dataset.location
    print(f"[Dataset] Downloaded to: {DATASET_ROOT}")

In [ ]:
# ====================================================================
# 📥 Option B: NOT NEEDED on Kaggle if you used "Add Input" -> Datasets
# ====================================================================
# Your dataset is already attached and mounted read-only under /kaggle/input/.
# No download or unzip step is required here. Skip running this cell.
USE_OWN_ZIP = False
print("[Kaggle] Using an attached Input dataset — skipping zip download/extraction.")


In [ ]:
# ====================================================================
# 📥 Kaggle: Point directly at your attached Input dataset
# ====================================================================
# Your dataset was added via "Add Input" and is named "YOLO_Fabric".
# Kaggle mounts it read-only at /kaggle/input/<dataset-name>/...
# Your zip's internal top-level folder was "YOLOv8", so the real path is:
DATASET_ROOT = "/kaggle/input/YOLO_Fabric/YOLOv8"

# Sanity check — adjust the path above if this fails (use the file browser
# in the left sidebar under /kaggle/input to confirm the exact folder name).
assert os.path.exists(DATASET_ROOT), (
    f"{DATASET_ROOT} not found. Open the left sidebar file browser, navigate "
    f"into /kaggle/input/YOLO_Fabric/, and update DATASET_ROOT to match the "
    f"folder that directly contains data.yaml."
)
print(f"[Dataset] Using {DATASET_ROOT}")


In [ ]:
# ====================================================================
# 🔍 Verify dataset layout & load class names from data.yaml
# ====================================================================
def discover_dataset(root):
    '''Finds data.yaml (or infers structure) and returns paths + class names.'''
    yaml_candidates = glob.glob(os.path.join(root, "**", "data.yaml"), recursive=True)
    if not yaml_candidates:
        raise FileNotFoundError(
            "No data.yaml found under the dataset root. Your dataset must include a "
            "data.yaml with 'names' and 'train'/'val' (or 'valid') image folder paths."
        )
    yaml_path = yaml_candidates[0]
    with open(yaml_path, 'r') as f:
        cfg = yaml.safe_load(f)

    base = os.path.dirname(yaml_path)
    def resolve(p):
        p = p.replace("../", "")
        cand = os.path.join(base, p)
        return cand if os.path.exists(cand) else p

    train_images = resolve(cfg.get('train', 'train/images'))
    val_key = 'val' if 'val' in cfg else 'valid'
    val_images = resolve(cfg.get(val_key, 'valid/images'))
    names = cfg['names']
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names.keys())]
    return train_images, val_images, names, yaml_path

assert 'DATASET_ROOT' in dir() and DATASET_ROOT is not None, \
    "Set DATASET_ROOT by running one of the dataset options above first."

TRAIN_IMG_DIR, VAL_IMG_DIR, CLASS_NAMES, YAML_PATH = discover_dataset(DATASET_ROOT)
NUM_CLASSES = len(CLASS_NAMES)

print(f"[data.yaml] {YAML_PATH}")
print(f"[Classes ({NUM_CLASSES})] {CLASS_NAMES}")
print(f"[Train images dir] {TRAIN_IMG_DIR}  -> {len(glob.glob(os.path.join(TRAIN_IMG_DIR, '*')))} files")
print(f"[Val images dir]   {VAL_IMG_DIR}  -> {len(glob.glob(os.path.join(VAL_IMG_DIR, '*')))} files")

## 🛠️ Section 2: Pre-processing & Data Augmentation (CLAHE + Mosaic)
Unchanged from the original design: CLAHE boosts low-contrast defect visibility, and mosaic augmentation helps small-object learning. These are now applied to **real images** loaded from disk.

In [ ]:
class FabricDataPreprocessor:
    '''
    Applies Contrast-Limited Adaptive Histogram Equalization (CLAHE)
    and Photometric/Geometric Data Augmentation for Fabric Defect Images.
    '''
    @staticmethod
    def apply_clahe(img_rgb: np.ndarray, clip_limit: float = 2.5, tile_grid_size: tuple = (8, 8)) -> np.ndarray:
        '''Applies CLAHE on L-channel of LAB color space to enhance defect contrast.'''
        lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
        cl = clahe.apply(l)
        limg = cv2.merge((cl, a, b))
        enhanced = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
        return enhanced

    @staticmethod
    def letterbox(img: np.ndarray, boxes: np.ndarray, target_size: int = 640):
        '''Resizes with aspect-ratio preserved padding; boxes are cx,cy,w,h normalized (0-1) and are unaffected
        since they stay normalized to the padded square canvas.'''
        h, w = img.shape[:2]
        scale = target_size / max(h, w)
        nh, nw = int(round(h * scale)), int(round(w * scale))
        resized = cv2.resize(img, (nw, nh))
        canvas = np.full((target_size, target_size, 3), 114, dtype=np.uint8)
        top = (target_size - nh) // 2
        left = (target_size - nw) // 2
        canvas[top:top + nh, left:left + nw] = resized

        if len(boxes) > 0:
            # Convert normalized (orig) -> pixel (orig) -> pixel (canvas) -> normalized (canvas)
            boxes = boxes.copy()
            boxes[:, 1] = (boxes[:, 1] * w * scale + left) / target_size
            boxes[:, 2] = (boxes[:, 2] * h * scale + top) / target_size
            boxes[:, 3] = (boxes[:, 3] * w * scale) / target_size
            boxes[:, 4] = (boxes[:, 4] * h * scale) / target_size
        return canvas, boxes

    @staticmethod
    def random_hflip(img: np.ndarray, boxes: np.ndarray, p: float = 0.5):
        if random.random() < p:
            img = img[:, ::-1, :].copy()
            if len(boxes) > 0:
                boxes = boxes.copy()
                boxes[:, 1] = 1.0 - boxes[:, 1]
        return img, boxes

    @staticmethod
    def mosaic(quad_images: list, img_size: int):
        '''Combines 4 (img, boxes) pairs into one img_size x img_size mosaic.
        Each source is CLAHE-processed but NOT yet letterboxed; this letterboxes each
        into its own quadrant (tile = img_size/2) and remaps boxes into full-canvas
        normalized coordinates. Increases effective training diversity 4x per sample.'''
        tile = img_size // 2
        canvas = np.full((img_size, img_size, 3), 114, dtype=np.uint8)
        offsets = [(0, 0), (tile, 0), (0, tile), (tile, tile)]
        all_boxes = []
        for (img, boxes), (ox, oy) in zip(quad_images, offsets):
            tile_img, tile_boxes = FabricDataPreprocessor.letterbox(img, boxes, tile)
            canvas[oy:oy + tile, ox:ox + tile] = tile_img
            if len(tile_boxes) > 0:
                b = tile_boxes.copy()
                b[:, 1] = (b[:, 1] * tile + ox) / img_size
                b[:, 2] = (b[:, 2] * tile + oy) / img_size
                b[:, 3] = b[:, 3] * tile / img_size
                b[:, 4] = b[:, 4] * tile / img_size
                all_boxes.append(b)
        boxes_out = np.concatenate(all_boxes, axis=0) if all_boxes else np.zeros((0, 5), dtype=np.float32)
        return canvas, boxes_out

    @staticmethod
    def random_rotate_scale(img: np.ndarray, boxes: np.ndarray, img_size: int,
                             degrees: float = 15.0, scale_range: tuple = (0.7, 1.3), p: float = 0.5):
        '''Random rotation + scale jitter around the canvas center, with correct box remapping
        (transforms all 4 box corners, then takes the new axis-aligned enclosing box).'''
        if random.random() > p:
            return img, boxes
        angle = random.uniform(-degrees, degrees)
        scale = random.uniform(*scale_range)
        center = (img_size / 2, img_size / 2)
        M = cv2.getRotationMatrix2D(center, angle, scale)
        img_t = cv2.warpAffine(img, M, (img_size, img_size), borderValue=(114, 114, 114))
        if len(boxes) == 0:
            return img_t, boxes

        b = boxes.copy()
        cx, cy, w, h = b[:, 1] * img_size, b[:, 2] * img_size, b[:, 3] * img_size, b[:, 4] * img_size
        x1, y1, x2, y2 = cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2

        new_rows = []
        for i in range(len(b)):
            corners = np.array([[x1[i], y1[i]], [x2[i], y1[i]], [x2[i], y2[i]], [x1[i], y2[i]]])
            ones = np.ones((4, 1))
            pts = np.hstack([corners, ones])
            transformed = (M @ pts.T).T
            nx1, ny1 = max(transformed[:, 0].min(), 0), max(transformed[:, 1].min(), 0)
            nx2, ny2 = min(transformed[:, 0].max(), img_size), min(transformed[:, 1].max(), img_size)
            if nx2 - nx1 < 2 or ny2 - ny1 < 2:
                continue  # box rotated/scaled out of frame or collapsed to near-zero area
            new_rows.append([b[i, 0], (nx1 + nx2) / 2 / img_size, (ny1 + ny2) / 2 / img_size,
                              (nx2 - nx1) / img_size, (ny2 - ny1) / img_size])
        boxes_out = np.array(new_rows, dtype=np.float32) if new_rows else np.zeros((0, 5), dtype=np.float32)
        return img_t, boxes_out

    @staticmethod
    def random_hsv(img: np.ndarray, h_gain: float = 0.015, s_gain: float = 0.5, v_gain: float = 0.3, p: float = 0.5):
        '''Random HSV color jitter — helps the model generalize across fabric color/lighting variation.'''
        if random.random() > p:
            return img
        img_hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
        img_hsv[..., 0] = (img_hsv[..., 0] + random.uniform(-h_gain, h_gain) * 179) % 179
        img_hsv[..., 1] = np.clip(img_hsv[..., 1] * random.uniform(1 - s_gain, 1 + s_gain), 0, 255)
        img_hsv[..., 2] = np.clip(img_hsv[..., 2] * random.uniform(1 - v_gain, 1 + v_gain), 0, 255)
        return cv2.cvtColor(img_hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

print("[Section 2] Preprocessor, CLAHE, letterbox, mosaic, rotation/scale, HSV jitter & flip augmentation loaded.")

## 🏗️ Section 3: Ghost-Lightweight Backbone (`GhostConv` & `C2fGhost`)
Unchanged — replaces standard heavy convolutions with **Ghost Modules** to cut parameter count while preserving feature capacity.

In [ ]:
class GhostConv(nn.Module):
    '''
    Ghost Convolution Block (Han et al. 2020).
    Splits output channels into primary conv features and cheap depthwise linear transform features.
    '''
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 1, ratio: int = 2, dw_kernel_size: int = 3, stride: int = 1, relu: bool = True):
        super(GhostConv, self).__init__()
        self.out_channels = out_channels
        init_channels = math.ceil(out_channels / ratio)
        new_channels = init_channels * (ratio - 1)

        self.primary_conv = nn.Sequential(
            nn.Conv2d(in_channels, init_channels, kernel_size, stride, kernel_size // 2, bias=False),
            nn.BatchNorm2d(init_channels),
            nn.SiLU() if relu else nn.Identity()
        )
        self.cheap_operation = nn.Sequential(
            nn.Conv2d(init_channels, new_channels, dw_kernel_size, 1, dw_kernel_size // 2, groups=init_channels, bias=False),
            nn.BatchNorm2d(new_channels),
            nn.SiLU() if relu else nn.Identity()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.primary_conv(x)
        x2 = self.cheap_operation(x1)
        out = torch.cat([x1, x2], dim=1)
        return out[:, :self.out_channels, :, :]


class C2fGhost(nn.Module):
    '''
    Ghost-based Cross-Stage Partial (CSP) Bottleneck module (C2fGhost).
    Concatenates intermediate feature maps from Ghost Bottlenecks for lightweight, rich gradient flow.
    '''
    def __init__(self, in_channels: int, out_channels: int, n: int = 1, shortcut: bool = True, g: int = 1, e: float = 0.5):
        super(C2fGhost, self).__init__()
        self.c = int(out_channels * e)
        self.cv1 = GhostConv(in_channels, 2 * self.c, kernel_size=1)
        self.cv2 = GhostConv((2 + n) * self.c, out_channels, kernel_size=1)
        self.m = nn.ModuleList([GhostConv(self.c, self.c, kernel_size=3) for _ in range(n)])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = list(self.cv1(x).chunk(2, 1))
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))

print("[Section 3] GhostConv & C2fGhost Modules compiled.")

## 👁️ Section 4: Attention-Guided Mechanisms (`Coordinate Attention` & `SimAM`)
Unchanged — Coordinate Attention encodes directional position (useful for long yarn-cut defects), SimAM is a parameter-free channel/spatial attention.

In [ ]:
class CoordinateAttention(nn.Module):
    '''
    Coordinate Attention Module (Hou et al. 2021).
    Encodes spatial position along horizontal (X) and vertical (Y) axes for fine linear fabric defects.
    '''
    def __init__(self, in_channels: int, out_channels: int, reduction: int = 32):
        super(CoordinateAttention, self).__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))

        mip = max(8, in_channels // reduction)
        self.conv1 = nn.Conv2d(in_channels, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.SiLU()

        self.conv_h = nn.Conv2d(mip, out_channels, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, out_channels, kernel_size=1, stride=1, padding=0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        n, c, h, w = x.size()
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)

        y = torch.cat([x_h, x_w], dim=2)
        y = self.act(self.bn1(self.conv1(y)))

        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)

        a_h = torch.sigmoid(self.conv_h(x_h))
        a_w = torch.sigmoid(self.conv_w(x_w))
        return identity * a_h * a_w


class SimAM(nn.Module):
    '''
    Parameter-free 3D Attention Module (SimAM - Yang et al. 2021).
    Derives 3D attention weights based on neuroscience energy function without extra parameters.
    '''
    def __init__(self, e_lambda: float = 1e-4):
        super(SimAM, self).__init__()
        self.e_lambda = e_lambda

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        n, c, h, w = x.size()
        n_pixels = h * w - 1
        d = (x - x.mean(dim=[2, 3], keepdim=True)).pow(2)
        v = d.sum(dim=[2, 3], keepdim=True) / n_pixels
        e_inv = d / (4 * (v + self.e_lambda)) + 0.5
        return x * torch.sigmoid(e_inv)

print("[Section 4] Coordinate Attention & SimAM Attention Modules compiled.")

## 🔀 Section 5: Multi-Scale Feature Fusion Neck (`BiFPN` + $160\times160$ Micro-Defect Layer)
Unchanged — weighted bidirectional feature pyramid across P2 (160x160), P3 (80x80), P4 (40x40), P5 (20x20).

In [ ]:
class BiFPNFeatureFusion(nn.Module):
    '''
    Weighted Bidirectional Feature Pyramid Network (BiFPN).
    Fuses multi-scale feature maps across P2 (160x160), P3 (80x80), P4 (40x40), and P5 (20x20).
    '''
    def __init__(self, in_channels_list: list = [64, 128, 256, 512], out_channels: int = 128):
        super(BiFPNFeatureFusion, self).__init__()
        c2, c3, c4, c5 = in_channels_list

        self.p2_proj = GhostConv(c2, out_channels, 1)
        self.p3_proj = GhostConv(c3, out_channels, 1)
        self.p4_proj = GhostConv(c4, out_channels, 1)
        self.p5_proj = GhostConv(c5, out_channels, 1)

        self.w_td4 = nn.Parameter(torch.ones(2, dtype=torch.float32))
        self.w_td3 = nn.Parameter(torch.ones(2, dtype=torch.float32))
        self.w_td2 = nn.Parameter(torch.ones(2, dtype=torch.float32))

        self.node_td4 = C2fGhost(out_channels, out_channels, n=1)
        self.node_td3 = C2fGhost(out_channels, out_channels, n=1)
        self.node_td2 = C2fGhost(out_channels, out_channels, n=1)

        self.ca_attn = CoordinateAttention(out_channels, out_channels)

    def forward(self, features: list) -> list:
        p2 = self.p2_proj(features[0])
        p3 = self.p3_proj(features[1])
        p4 = self.p4_proj(features[2])
        p5 = self.p5_proj(features[3])

        w_td4 = F.relu(self.w_td4) / (torch.sum(F.relu(self.w_td4)) + 1e-4)
        p5_up = F.interpolate(p5, size=p4.shape[2:], mode='nearest')
        p4_td = self.node_td4(w_td4[0] * p4 + w_td4[1] * p5_up)

        w_td3 = F.relu(self.w_td3) / (torch.sum(F.relu(self.w_td3)) + 1e-4)
        p4_up = F.interpolate(p4_td, size=p3.shape[2:], mode='nearest')
        p3_td = self.node_td3(w_td3[0] * p3 + w_td3[1] * p4_up)

        w_td2 = F.relu(self.w_td2) / (torch.sum(F.relu(self.w_td2)) + 1e-4)
        p3_up = F.interpolate(p3_td, size=p2.shape[2:], mode='nearest')
        p2_out = self.node_td2(w_td2[0] * p2 + w_td2[1] * p3_up)
        p2_out = self.ca_attn(p2_out)

        return [p2_out, p3_td, p4_td, p5]

print("[Section 5] BiFPN Multi-Scale Feature Fusion with P2 (160x160) layer compiled.")

## 🎯 Section 6: Lightweight Shared Head, Box Decoding, SIoU Loss & Real Target Assignment
This is the part that was previously fake. The head itself is unchanged, but we now add:
- **Box decoding**: converts raw `(tx, ty, tw, th)` predictions per grid cell into actual image-space boxes using the cell's stride, the way YOLO/FCOS-style anchor-free detectors do.
- **Center-sampling target assignment**: for every ground-truth box, the grid cells whose center falls inside a shrunk region of the box (at the scale whose stride best matches the box size) are marked positive.
- **A real composite loss**: SIoU for box regression (already defined, previously unused) + focal loss for objectness + BCE for classification, computed only where it should be, i.e. against real targets instead of `mean(abs(output))`.

In [ ]:
class LightweightSharedHead(nn.Module):
    '''
    Lightweight Shared-Convolution Head across multi-scale feature outputs (P2, P3, P4, P5).
    Uses Group Normalization and learnable per-scale scale layers.
    '''
    def __init__(self, num_classes: int = 6, in_channels: int = 128, strides: list = [4, 8, 16, 32]):
        super(LightweightSharedHead, self).__init__()
        self.num_classes = num_classes
        self.strides = strides

        self.shared_cls_conv = nn.Sequential(
            GhostConv(in_channels, in_channels, kernel_size=3),
            nn.GroupNorm(8, in_channels),
            nn.SiLU()
        )
        self.shared_reg_conv = nn.Sequential(
            GhostConv(in_channels, in_channels, kernel_size=3),
            nn.GroupNorm(8, in_channels),
            nn.SiLU()
        )

        self.cls_head = nn.Conv2d(in_channels, num_classes, kernel_size=1)
        self.reg_head = nn.Conv2d(in_channels, 4, kernel_size=1)
        self.obj_head = nn.Conv2d(in_channels, 1, kernel_size=1)

        # Learnable per-scale distance scale (standard FCOS-style trick so one reg_head
        # can be shared across scales of very different stride).
        self.scale_layers = nn.ParameterList([nn.Parameter(torch.tensor(1.0)) for _ in strides])

    def forward(self, features: list) -> list:
        outputs = []
        for i, feat in enumerate(features):
            cls_feat = self.shared_cls_conv(feat)
            reg_feat = self.shared_reg_conv(feat)

            cls_out = self.cls_head(cls_feat)
            reg_out = self.reg_head(reg_feat) * self.scale_layers[i]
            obj_out = self.obj_head(reg_feat)

            outputs.append({
                'cls': cls_out,   # (B, num_classes, H, W) raw logits
                'reg': reg_out,   # (B, 4, H, W) raw (l, t, r, b) distances in stride units (pre-exp)
                'obj': obj_out,   # (B, 1, H, W) raw objectness logits
                'stride': self.strides[i]
            })
        return outputs


def compute_siou_loss(pred_boxes: torch.Tensor, target_boxes: torch.Tensor) -> torch.Tensor:
    '''
    SIoU (SCYLLA-IoU) Loss (Zheng et al. 2022). Boxes are (x1, y1, x2, y2).
    Includes Angle Cost, Distance Cost, Shape Cost, and IoU Cost.
    '''
    b1_x1, b1_y1, b1_x2, b1_y2 = pred_boxes[:, 0], pred_boxes[:, 1], pred_boxes[:, 2], pred_boxes[:, 3]
    b2_x1, b2_y1, b2_x2, b2_y2 = target_boxes[:, 0], target_boxes[:, 1], target_boxes[:, 2], target_boxes[:, 3]

    cw = torch.max(b1_x2, b2_x2) - torch.min(b1_x1, b2_x1) + 1e-7
    ch = torch.max(b1_y2, b2_y2) - torch.min(b1_y1, b2_y1) + 1e-7

    px, py = (b1_x1 + b1_x2) / 2.0, (b1_y1 + b1_y2) / 2.0
    tx, ty = (b2_x1 + b2_x2) / 2.0, (b2_y1 + b2_y2) / 2.0

    sigma = torch.pow((tx - px) ** 2 + (ty - py) ** 2, 0.5) + 1e-7
    sin_alpha = torch.abs(ty - py) / sigma
    sin_beta = torch.abs(tx - px) / sigma
    sin_alpha = torch.where(sin_alpha > math.sin(math.pi / 4), sin_beta, sin_alpha)

    angle_cost = 1.0 - 2.0 * (torch.pow(torch.sin(torch.arcsin(sin_alpha.clamp(-1 + 1e-7, 1 - 1e-7)) - math.pi / 4), 2))
    rho_x = ((tx - px) / cw) ** 2
    rho_y = ((ty - py) / ch) ** 2
    gamma = 2.0 - angle_cost
    distance_cost = 2.0 - torch.exp(-gamma * rho_x) - torch.exp(-gamma * rho_y)

    inter = (torch.min(b1_x2, b2_x2) - torch.max(b1_x1, b2_x1)).clamp(min=0) * \
            (torch.min(b1_y2, b2_y2) - torch.max(b1_y1, b2_y1)).clamp(min=0)
    area1 = (b1_x2 - b1_x1) * (b1_y2 - b1_y1)
    area2 = (b2_x2 - b2_x1) * (b2_y2 - b2_y1)
    iou = inter / (area1 + area2 - inter + 1e-7)

    siou = iou - 0.5 * (distance_cost + 0.5)
    return 1.0 - siou.clamp(min=-1.0, max=1.0), iou.detach()


def decode_predictions(head_outputs: list, img_size: int):
    '''Converts every scale's raw (l,t,r,b) distances into absolute xyxy boxes in pixel space.
    Returns a list (per scale) of dicts with 'boxes' (B,H*W,4), 'obj' (B,H*W), 'cls' (B,H*W,C), 'grid_xy' (H*W,2), 'stride'.'''
    decoded = []
    for out in head_outputs:
        B, C, H, W = out['cls'].shape
        stride = out['stride']
        yv, xv = torch.meshgrid(torch.arange(H, device=out['cls'].device),
                                 torch.arange(W, device=out['cls'].device), indexing='ij')
        grid_xy = (torch.stack([xv, yv], dim=-1).float() + 0.5) * stride  # (H, W, 2) cell centers in pixels
        grid_xy_flat = grid_xy.view(-1, 2)

        reg = out['reg'].permute(0, 2, 3, 1).reshape(B, H * W, 4)
        ltrb = F.softplus(reg) * stride  # ensure positive distances, scaled to pixels

        cx = grid_xy_flat[None, :, 0]
        cy = grid_xy_flat[None, :, 1]
        x1 = cx - ltrb[..., 0]
        y1 = cy - ltrb[..., 1]
        x2 = cx + ltrb[..., 2]
        y2 = cy + ltrb[..., 3]
        boxes = torch.stack([x1, y1, x2, y2], dim=-1).clamp(0, img_size)

        obj = out['obj'].permute(0, 2, 3, 1).reshape(B, H * W)
        cls = out['cls'].permute(0, 2, 3, 1).reshape(B, H * W, C)

        decoded.append({'boxes': boxes, 'obj': obj, 'cls': cls, 'grid_xy': grid_xy_flat, 'stride': stride})
    return decoded


class AGLYOLOLoss(nn.Module):
    '''
    Real anchor-free detection loss:
      - Center-sampling assignment of each GT box to grid cells at the most appropriate FPN scale
      - SIoU loss for box regression (positives only)
      - Focal loss for objectness (all cells, heavy imbalance toward negatives)
      - BCE loss for classification (positives only)
    '''
    def __init__(self, num_classes: int, img_size: int = 640, center_radius: float = 2.5,
                 obj_weight: float = 1.0, box_weight: float = 5.0, cls_weight: float = 1.0,
                 focal_gamma: float = 2.0, focal_alpha: float = 0.25):
        super().__init__()
        self.num_classes = num_classes
        self.img_size = img_size
        self.center_radius = center_radius
        self.obj_weight = obj_weight
        self.box_weight = box_weight
        self.cls_weight = cls_weight
        self.focal_gamma = focal_gamma
        self.focal_alpha = focal_alpha

    def focal_bce(self, logits, targets):
        p = torch.sigmoid(logits)
        ce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.focal_alpha * targets + (1 - self.focal_alpha) * (1 - targets)
        loss = alpha_t * (1 - p_t) ** self.focal_gamma * ce
        return loss

    def forward(self, head_outputs: list, targets: list):
        '''targets: list (len B) of Tensor (N_i, 5) = [cls_id, x1, y1, x2, y2] in PIXEL coords (0..img_size).'''
        decoded = decode_predictions(head_outputs, self.img_size)
        device = head_outputs[0]['cls'].device
        B = head_outputs[0]['cls'].shape[0]

        total_obj_loss = torch.zeros((), device=device)
        total_box_loss = torch.zeros((), device=device)
        total_cls_loss = torch.zeros((), device=device)
        num_pos_total = 0

        for scale in decoded:
            grid_xy = scale['grid_xy']            # (HW, 2)
            stride = scale['stride']
            HW = grid_xy.shape[0]

            obj_targets = torch.zeros((B, HW), device=device)
            cls_targets = torch.zeros((B, HW, self.num_classes), device=device)
            box_targets = torch.zeros((B, HW, 4), device=device)
            pos_masks = torch.zeros((B, HW), dtype=torch.bool, device=device)

            for b in range(B):
                gt = targets[b]
                if gt.numel() == 0:
                    continue
                gt_cls = gt[:, 0].long()
                gt_boxes = gt[:, 1:5]  # x1,y1,x2,y2 pixels
                gt_w = gt_boxes[:, 2] - gt_boxes[:, 0]
                gt_h = gt_boxes[:, 3] - gt_boxes[:, 1]
                gt_cx = (gt_boxes[:, 0] + gt_boxes[:, 2]) / 2
                gt_cy = (gt_boxes[:, 1] + gt_boxes[:, 3]) / 2
                gt_scale_metric = torch.sqrt(gt_w * gt_h)

                # Only assign a GT box to this FPN scale if its size roughly matches the stride's
                # receptive field range (standard multi-scale FPN assignment heuristic).
                lo, hi = stride * 1.0, stride * 8.0
                if stride == min(s['stride'] for s in decoded):
                    lo = 0.0  # smallest stride also catches tiny objects below 1x stride
                if stride == max(s['stride'] for s in decoded):
                    hi = float('inf')  # largest stride also catches huge objects
                scale_ok = (gt_scale_metric >= lo) & (gt_scale_metric < hi)
                if scale_ok.sum() == 0:
                    continue

                idxs = scale_ok.nonzero(as_tuple=True)[0]
                for gi in idxs.tolist():
                    cx, cy = gt_cx[gi], gt_cy[gi]
                    radius = self.center_radius * stride
                    dist = ((grid_xy[:, 0] - cx) ** 2 + (grid_xy[:, 1] - cy) ** 2).sqrt()
                    inside = (dist < radius) & \
                             (grid_xy[:, 0] > gt_boxes[gi, 0]) & (grid_xy[:, 0] < gt_boxes[gi, 2]) & \
                             (grid_xy[:, 1] > gt_boxes[gi, 1]) & (grid_xy[:, 1] < gt_boxes[gi, 3])
                    if inside.sum() == 0:
                        continue
                    obj_targets[b, inside] = 1.0
                    cls_targets[b, inside, gt_cls[gi]] = 1.0
                    box_targets[b, inside] = gt_boxes[gi]
                    pos_masks[b, inside] = True

            # --- Objectness (focal, all cells) ---
            obj_loss = self.focal_bce(scale['obj'], obj_targets).mean()
            total_obj_loss = total_obj_loss + obj_loss

            n_pos = int(pos_masks.sum().item())
            num_pos_total += n_pos
            if n_pos > 0:
                pred_boxes_pos = scale['boxes'][pos_masks]
                target_boxes_pos = box_targets[pos_masks]
                siou, _ = compute_siou_loss(pred_boxes_pos, target_boxes_pos)
                total_box_loss = total_box_loss + siou.mean()

                cls_logits_pos = scale['cls'][pos_masks]
                cls_targets_pos = cls_targets[pos_masks]
                total_cls_loss = total_cls_loss + F.binary_cross_entropy_with_logits(
                    cls_logits_pos, cls_targets_pos, reduction='mean')

        total_loss = (self.obj_weight * total_obj_loss +
                      self.box_weight * total_box_loss +
                      self.cls_weight * total_cls_loss)
        return total_loss, {
            'obj_loss': total_obj_loss.item(),
            'box_loss': total_box_loss.item(),
            'cls_loss': total_cls_loss.item(),
            'num_positives': num_pos_total
        }

print("[Section 6] Head, box decoding, SIoU loss, and real anchor-free target assignment compiled.")

## ⚡ Section 7: Complete AGL-YOLO Framework Architecture Assembly
Unchanged — assembles the Ghost backbone, BiFPN neck, and shared head into Nano / Small / Medium variants.

In [ ]:
class AGLYOLO_Backbone(nn.Module):
    '''Ghost-convolution backbone extracting features at P2, P3, P4, and P5.'''
    def __init__(self, in_channels: int = 3, width_mult: float = 1.0):
        super(AGLYOLO_Backbone, self).__init__()
        c = lambda ch: max(16, int(ch * width_mult))

        self.stem = nn.Sequential(
            GhostConv(in_channels, c(16), 3, stride=2),
            GhostConv(c(16), c(32), 3, stride=1)
        )
        self.stage_p2 = nn.Sequential(GhostConv(c(32), c(64), 3, stride=2), C2fGhost(c(64), c(64), n=1))
        self.stage_p3 = nn.Sequential(GhostConv(c(64), c(128), 3, stride=2), C2fGhost(c(128), c(128), n=2))
        self.stage_p4 = nn.Sequential(GhostConv(c(128), c(256), 3, stride=2), C2fGhost(c(256), c(256), n=2))
        self.stage_p5 = nn.Sequential(GhostConv(c(256), c(512), 3, stride=2), C2fGhost(c(512), c(512), n=1), CoordinateAttention(c(512), c(512)))
        self.out_channels = [c(64), c(128), c(256), c(512)]

    def forward(self, x: torch.Tensor) -> list:
        x = self.stem(x)
        p2 = self.stage_p2(x)
        p3 = self.stage_p3(p2)
        p4 = self.stage_p4(p3)
        p5 = self.stage_p5(p4)
        return [p2, p3, p4, p5]


class AGLYOLO(nn.Module):
    '''AGL-YOLO Full Network Architecture.'''
    def __init__(self, num_classes: int = 6, width_mult: float = 1.0, img_size: int = 640):
        super(AGLYOLO, self).__init__()
        self.num_classes = num_classes
        self.img_size = img_size
        self.backbone = AGLYOLO_Backbone(3, width_mult=width_mult)
        b_channels = self.backbone.out_channels
        neck_channels = max(64, int(128 * width_mult))

        self.neck = BiFPNFeatureFusion(in_channels_list=b_channels, out_channels=neck_channels)
        self.head = LightweightSharedHead(num_classes=num_classes, in_channels=neck_channels, strides=[4, 8, 16, 32])

    def forward(self, x: torch.Tensor):
        b_feats = self.backbone(x)
        n_feats = self.neck(b_feats)
        outputs = self.head(n_feats)
        return outputs

    def get_attention_maps(self, x: torch.Tensor) -> torch.Tensor:
        '''Extracts coordinate attention spatial heatmap.'''
        b_feats = self.backbone(x)
        n_feats = self.neck(b_feats)
        p2_feat = n_feats[0]
        heatmap = torch.mean(torch.abs(p2_feat), dim=1, keepdim=True)
        heatmap = F.interpolate(heatmap, size=x.shape[2:], mode='bilinear', align_corners=False)
        min_val, max_val = heatmap.min(), heatmap.max()
        return (heatmap - min_val) / (max_val - min_val + 1e-6)

    def num_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Presets
def create_aglyolo_nano(num_classes: int = 6, img_size: int = 640) -> AGLYOLO:
    return AGLYOLO(num_classes, width_mult=0.5, img_size=img_size)

def create_aglyolo_small(num_classes: int = 6, img_size: int = 640) -> AGLYOLO:
    return AGLYOLO(num_classes, width_mult=0.75, img_size=img_size)

def create_aglyolo_medium(num_classes: int = 6, img_size: int = 640) -> AGLYOLO:
    return AGLYOLO(num_classes, width_mult=1.0, img_size=img_size)

print("[Section 7] AGL-YOLO Framework Architecture Variants (Nano, Small, Medium) initialized.")

## 📊 Section 8: Real Fabric Defect Dataset Loader
Loads actual images and YOLO-format `.txt` labels from `TRAIN_IMG_DIR` / `VAL_IMG_DIR` (set in Section 1). Each label file has one line per box: `class_id cx cy w h` (all normalized 0-1, matching Ultralytics/YOLO convention).

In [ ]:
IMG_SIZE = 640

def find_label_path(img_path: str) -> str:
    '''Standard YOLO convention: .../images/xxx.jpg -> .../labels/xxx.txt'''
    return img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'


class YOLOFabricDataset(Dataset):
    '''Loads real images + YOLO-format labels for fabric defect detection.
    Training augmentation: mosaic (4-image combine) -> random rotation/scale ->
    random horizontal flip -> random HSV jitter. Validation: CLAHE + letterbox only.'''
    def __init__(self, img_dir: str, img_size: int = 640, augment: bool = True, mosaic_prob: float = 0.5):
        self.img_paths = sorted(
            glob.glob(os.path.join(img_dir, '*.jpg')) +
            glob.glob(os.path.join(img_dir, '*.jpeg')) +
            glob.glob(os.path.join(img_dir, '*.png'))
        )
        if len(self.img_paths) == 0:
            raise FileNotFoundError(f"No images found in {img_dir}. Check DATASET_ROOT / data.yaml paths.")
        self.img_size = img_size
        self.augment = augment
        self.mosaic_prob = mosaic_prob if augment else 0.0

    def __len__(self):
        return len(self.img_paths)

    def _load_labels(self, label_path: str) -> np.ndarray:
        if not os.path.exists(label_path):
            return np.zeros((0, 5), dtype=np.float32)
        rows = []
        with open(label_path, 'r') as f:
            for line in f.read().strip().splitlines():
                if not line.strip():
                    continue
                parts = line.split()
                cls_id, cx, cy, w, h = (float(x) for x in parts[:5])
                rows.append([cls_id, cx, cy, w, h])
        return np.array(rows, dtype=np.float32) if rows else np.zeros((0, 5), dtype=np.float32)

    def _load_raw(self, idx: int):
        '''Loads + CLAHE-enhances one source image at its native resolution (no resize yet).'''
        img_path = self.img_paths[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        boxes = self._load_labels(find_label_path(img_path))
        img = FabricDataPreprocessor.apply_clahe(img)
        return img, boxes

    def __getitem__(self, idx: int):
        if random.random() < self.mosaic_prob:
            idxs = [idx] + [random.randint(0, len(self) - 1) for _ in range(3)]
            quad = [self._load_raw(i) for i in idxs]
            img, boxes = FabricDataPreprocessor.mosaic(quad, self.img_size)
        else:
            img, boxes = self._load_raw(idx)
            img, boxes = FabricDataPreprocessor.letterbox(img, boxes, self.img_size)

        if self.augment:
            img, boxes = FabricDataPreprocessor.random_rotate_scale(img, boxes, self.img_size)
            img, boxes = FabricDataPreprocessor.random_hflip(img, boxes)
            img = FabricDataPreprocessor.random_hsv(img)

        # Convert normalized cx,cy,w,h -> pixel xyxy for the loss function
        if len(boxes) > 0:
            cls_id = boxes[:, 0]
            cx, cy, w, h = boxes[:, 1] * self.img_size, boxes[:, 2] * self.img_size, boxes[:, 3] * self.img_size, boxes[:, 4] * self.img_size
            x1, y1 = cx - w / 2, cy - h / 2
            x2, y2 = cx + w / 2, cy + h / 2
            target = np.stack([cls_id, x1, y1, x2, y2], axis=1)
        else:
            target = np.zeros((0, 5), dtype=np.float32)

        tensor_in = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        return tensor_in, torch.from_numpy(target).float(), img


def yolo_collate_fn(batch):
    imgs, targets, raws = zip(*batch)
    imgs = torch.stack(imgs, dim=0)
    return imgs, list(targets), list(raws)   # targets stays a list since box count varies per image


train_ds = YOLOFabricDataset(TRAIN_IMG_DIR, img_size=IMG_SIZE, augment=True)
val_ds = YOLOFabricDataset(VAL_IMG_DIR, img_size=IMG_SIZE, augment=False)

BATCH_SIZE = 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=yolo_collate_fn,
                           num_workers=2, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=yolo_collate_fn,
                         num_workers=2)

print(f"[Dataset] {len(train_ds)} train images, {len(val_ds)} val images, {NUM_CLASSES} classes: {CLASS_NAMES}")

## 🧹 Section 8.5: Post-processing (NMS) & mAP Evaluation Function
Defined here (before training) so it can be used for **mAP-based checkpointing** during training, not just at the end.

⚠️ **Fix applied:** `evaluate_map` now runs NMS at a very low confidence threshold (`eval_conf_thresh=0.001`) before handing predictions to `torchmetrics`. mAP is defined as the area under the full precision-recall curve — pre-filtering at a high confidence threshold (e.g. 0.25) before computing it throws away lower-confidence true positives and artificially collapses the score. The 0.25 threshold is still used, correctly, for the human-facing visualization in Section 13 — just not for the metric itself.

In [ ]:
def postprocess_batch(head_outputs, img_size, conf_thresh=0.25, iou_thresh=0.45, max_det=300):
    '''Returns a list (per image in batch) of dict(boxes=(N,4), scores=(N,), labels=(N,)).
    conf_thresh here is a DISPLAY/deployment threshold — use a much lower value for mAP evaluation
    (see evaluate_map below), since mAP needs the full precision-recall curve, not one operating point.'''
    decoded = decode_predictions(head_outputs, img_size)
    B = head_outputs[0]['cls'].shape[0]
    results = []
    for b in range(B):
        all_boxes, all_scores, all_labels = [], [], []
        for scale in decoded:
            obj = torch.sigmoid(scale['obj'][b])                 # (HW,)
            cls_probs = torch.sigmoid(scale['cls'][b])            # (HW, C)
            scores_per_cls, labels_per_cls = cls_probs.max(dim=1)
            scores = obj * scores_per_cls
            keep = scores > conf_thresh
            if keep.sum() == 0:
                continue
            all_boxes.append(scale['boxes'][b][keep])
            all_scores.append(scores[keep])
            all_labels.append(labels_per_cls[keep])

        if len(all_boxes) == 0:
            results.append({'boxes': torch.zeros((0, 4)), 'scores': torch.zeros((0,)), 'labels': torch.zeros((0,), dtype=torch.long)})
            continue

        boxes = torch.cat(all_boxes); scores = torch.cat(all_scores); labels = torch.cat(all_labels)
        final_boxes, final_scores, final_labels = [], [], []
        for c in labels.unique():
            m = labels == c
            keep_idx = torch.ops.torchvision.nms(boxes[m], scores[m], iou_thresh) if hasattr(torch.ops, 'torchvision') else None
            if keep_idx is None:
                import torchvision
                keep_idx = torchvision.ops.nms(boxes[m], scores[m], iou_thresh)
            final_boxes.append(boxes[m][keep_idx])
            final_scores.append(scores[m][keep_idx])
            final_labels.append(labels[m][keep_idx])
        boxes = torch.cat(final_boxes); scores = torch.cat(final_scores); labels = torch.cat(final_labels)
        if len(scores) > max_det:
            top = scores.topk(max_det).indices
            boxes, scores, labels = boxes[top], scores[top], labels[top]
        results.append({'boxes': boxes.cpu(), 'scores': scores.cpu(), 'labels': labels.cpu()})
    return results


@torch.no_grad()
def evaluate_map(model, loader, img_size, device, eval_conf_thresh: float = 0.001, iou_thresh: float = 0.45, desc: str = "Evaluating mAP"):
    '''Computes real mAP@0.5, mAP@0.5:0.95, mAR@100 via torchmetrics.
    Uses a near-zero confidence threshold internally — NOT the 0.25 display threshold — so the
    full precision-recall curve is available to the metric. This is the standard, correct way
    to evaluate mAP (matches how pycocotools / ultralytics / torchmetrics expect predictions).'''
    was_training = model.training
    model.eval()
    metric = MeanAveragePrecision(box_format='xyxy', class_metrics=False)
    for imgs, targets, _ in tqdm(loader, desc=desc, leave=False):
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = postprocess_batch(outputs, img_size, conf_thresh=eval_conf_thresh, iou_thresh=iou_thresh)

        preds_formatted = [{'boxes': p['boxes'], 'scores': p['scores'], 'labels': p['labels']} for p in preds]
        gts_formatted = []
        for t in targets:
            if t.numel() == 0:
                gts_formatted.append({'boxes': torch.zeros((0, 4)), 'labels': torch.zeros((0,), dtype=torch.long)})
            else:
                gts_formatted.append({'boxes': t[:, 1:5].cpu(), 'labels': t[:, 0].long().cpu()})
        metric.update(preds_formatted, gts_formatted)

    result = metric.compute()
    if was_training:
        model.train()
    return result

print("[Section 8.5] NMS post-processing and correct-threshold mAP evaluation function ready.")

## 🚀 Section 9: Training Loop (Real Loss, Real Data)
Trains AGL-YOLO with the composite SIoU + focal-objectness + BCE-classification loss from Section 6, against your real dataset (now with mosaic/rotation/HSV augmentation).

⚠️ **Fixes applied here:**
- **LR warmup** (3 epochs, linear ramp) before the cosine schedule, so the untrained detection head doesn't destabilize the backbone early on.
- **mAP-based checkpointing**: every `EVAL_EVERY` epochs, real validation mAP@0.5 is computed (using the corrected low-threshold `evaluate_map` from Section 8.5) and the checkpoint is saved on **best mAP**, not best val loss — these can diverge.
- **Early stopping**: training stops if mAP hasn't improved for `EARLY_STOP_PATIENCE` evaluation cycles, so a long run doesn't waste GPU quota after it's plateaued.

In [ ]:
MODEL_SIZE = 'small'   # 'nano' | 'small' | 'medium'
NUM_EPOCHS = 150             # more epochs now that early stopping can cut it short safely
LEARNING_RATE = 1e-3
WARMUP_EPOCHS = 3
EVAL_EVERY = 5               # compute real val mAP every N epochs (mAP eval is slower than a loss pass)
EARLY_STOP_PATIENCE = 6      # stop if mAP hasn't improved for this many EVAL_EVERY cycles

model_builders = {'nano': create_aglyolo_nano, 'small': create_aglyolo_small, 'medium': create_aglyolo_medium}
model = model_builders[MODEL_SIZE](num_classes=NUM_CLASSES, img_size=IMG_SIZE).to(device)
print(f"[Model Stats] Variant: AGL-YOLO-{MODEL_SIZE} | Trainable Parameters: {model.num_parameters():,} ({model.num_parameters()/1e6:.2f} M)")

criterion = AGLYOLOLoss(num_classes=NUM_CLASSES, img_size=IMG_SIZE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, (NUM_EPOCHS - WARMUP_EPOCHS))
    return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

history = {'train_loss': [], 'val_loss': [], 'eval_epochs': [], 'map50': []}
best_map50 = 0.0
patience_counter = 0
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss, n_batches = 0.0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]")
    for imgs, targets, _ in pbar:
        imgs = imgs.to(device)
        targets = [t.to(device) for t in targets]

        optimizer.zero_grad()
        outputs = model(imgs)
        loss, parts = criterion(outputs, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()

        running_loss += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f"{loss.item():.3f}", pos=parts['num_positives'])

    train_loss = running_loss / max(n_batches, 1)
    scheduler.step()

    model.eval()
    val_running, v_batches = 0.0, 0
    with torch.no_grad():
        for imgs, targets, _ in val_loader:
            imgs = imgs.to(device)
            targets = [t.to(device) for t in targets]
            outputs = model(imgs)
            loss, _ = criterion(outputs, targets)
            val_running += loss.item()
            v_batches += 1
    val_loss = val_running / max(v_batches, 1)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    print(f"Epoch [{epoch}/{NUM_EPOCHS}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

    if epoch % EVAL_EVERY == 0 or epoch == NUM_EPOCHS:
        map_res = evaluate_map(model, val_loader, IMG_SIZE, device, eval_conf_thresh=0.001, desc=f"mAP @ epoch {epoch}")
        current_map50 = map_res['map_50'].item() * 100
        history['eval_epochs'].append(epoch)
        history['map50'].append(current_map50)
        print(f"  -> Val mAP@0.5: {current_map50:.2f}% (best so far: {best_map50:.2f}%)")

        if current_map50 > best_map50:
            best_map50 = current_map50
            patience_counter = 0
            torch.save(model.state_dict(), f'/kaggle/working/checkpoints/aglyolo_{MODEL_SIZE}_best.pt')
            print(f"  -> New best checkpoint saved (mAP@0.5={best_map50:.2f}%)")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"[Early stopping] No mAP@0.5 improvement for {EARLY_STOP_PATIENCE * EVAL_EVERY} epochs. Stopping at epoch {epoch}.")
                break

print(f"[Training Complete] Best val mAP@0.5: {best_map50:.2f}% | Checkpoint: /kaggle/working/checkpoints/aglyolo_{MODEL_SIZE}_best.pt")

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(history['train_loss'], label='Train Loss', color='tab:blue')
ax1.plot(history['val_loss'], label='Val Loss', color='tab:orange')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(loc='upper left'); ax1.grid(alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(history['eval_epochs'], history['map50'], label='Val mAP@0.5 (%)', color='tab:green', marker='o')
ax2.set_ylabel('mAP@0.5 (%)')
ax2.legend(loc='upper right')

plt.title('AGL-YOLO Training Curve (loss + real mAP@0.5)')
plt.tight_layout(); plt.show()

## 📋 Section 10: Final Evaluation on the Best Checkpoint
Loads the best checkpoint saved during training (by mAP, not loss — see Section 9) and reports final mAP@0.5, mAP@0.5:0.95, and mAR@100 using the corrected low-threshold evaluation from Section 8.5.

In [ ]:
best_ckpt = f'/kaggle/working/checkpoints/aglyolo_{MODEL_SIZE}_best.pt'
if os.path.exists(best_ckpt):
    model.load_state_dict(torch.load(best_ckpt, map_location=device))
    print(f"[Loaded] {best_ckpt}")
else:
    print("[WARNING] No checkpoint found — using the in-memory model from the end of training.")

map_results = evaluate_map(model, val_loader, IMG_SIZE, device, eval_conf_thresh=0.001, desc="Final evaluation")
print("\n=== Real Validation Metrics (AGL-YOLO, this run) ===")
print(f"mAP@0.5      : {map_results['map_50'].item()*100:.2f}%")
print(f"mAP@0.5:0.95 : {map_results['map'].item()*100:.2f}%")
print(f"mAR@100      : {map_results['mar_100'].item()*100:.2f}%")

## ⏱️ Section 11: Real FPS & GFLOPs Measurement
Measures actual inference speed on your hardware (GPU if available) and real FLOPs via `thop` — replacing the invented numbers from the demo table.

In [ ]:
from thop import profile

model.eval()
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)

with torch.no_grad():
    macs, params = profile(model, inputs=(dummy_input,), verbose=False)
gflops = macs * 2 / 1e9  # MACs -> FLOPs
print(f"[AGL-YOLO-{MODEL_SIZE}] Params: {params/1e6:.2f} M | GFLOPs: {gflops:.2f} @ {IMG_SIZE}x{IMG_SIZE}")

# Warm-up
with torch.no_grad():
    for _ in range(10):
        _ = model(dummy_input)
    if device.type == 'cuda':
        torch.cuda.synchronize()

    n_runs = 100
    start = time.time()
    for _ in range(n_runs):
        _ = model(dummy_input)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.time() - start

fps = n_runs / elapsed
print(f"[AGL-YOLO-{MODEL_SIZE}] Measured inference speed: {fps:.1f} FPS on {device} (batch size 1, {IMG_SIZE}x{IMG_SIZE})")
print("NOTE: FPS depends heavily on the actual GPU/CPU this cell runs on — quote the hardware alongside this number.")

## 📈 Section 12 (Optional): Real YOLOv8n Baseline for a Fair Comparison
Trains an actual YOLOv8n on the *same* dataset via `ultralytics`, so the comparison table is real numbers from real runs rather than numbers copied from other papers (which used different data/hardware/hyperparameters and are not a fair apples-to-apples comparison to your model).

In [ ]:
RUN_BASELINE_COMPARISON = False   # <-- set True once your AGL-YOLO run above works end-to-end

if RUN_BASELINE_COMPARISON:
    from ultralytics import YOLO

    baseline = YOLO('yolov8n.pt')
    baseline_results = baseline.train(data=YAML_PATH, epochs=NUM_EPOCHS, imgsz=IMG_SIZE, batch=BATCH_SIZE, verbose=False)
    baseline_metrics = baseline.val(data=YAML_PATH, imgsz=IMG_SIZE)

    yolov8n_map50 = baseline_metrics.box.map50 * 100
    yolov8n_map = baseline_metrics.box.map * 100
    yolov8n_params = sum(p.numel() for p in baseline.model.parameters()) / 1e6

    print(f"[YOLOv8n baseline] Params: {yolov8n_params:.2f}M | mAP@0.5: {yolov8n_map50:.2f}% | mAP@0.5:0.95: {yolov8n_map:.2f}%")

    comparison_table = [
        {"Model": "YOLOv8n (baseline, this run)", "Params (M)": round(yolov8n_params, 2), "mAP@0.5": round(yolov8n_map50, 2), "mAP@0.5:0.95": round(yolov8n_map, 2)},
        {"Model": f"AGL-YOLO-{MODEL_SIZE} (ours, this run)", "Params (M)": round(params/1e6, 2), "mAP@0.5": round(map_results['map_50'].item()*100, 2), "mAP@0.5:0.95": round(map_results['map'].item()*100, 2)},
    ]
    print("\n=== Real Comparison Table (both trained on your dataset, this session) ===")
    for row in comparison_table:
        print(row)
else:
    print("Set RUN_BASELINE_COMPARISON = True to train a real YOLOv8n baseline for comparison.")

## 🔍 Section 13: Defect Detection Visualizer & Coordinate Attention Heatmap
Runs the trained model on a real validation image and shows the input, the Coordinate Attention heatmap, and the actual detected boxes (after confidence thresholding + NMS) with real predicted class names and confidences.

In [ ]:
sample_idx = 0
tensor_in, gt_target, orig_np = val_ds[sample_idx]

model.eval()
with torch.no_grad():
    tensor_batch = tensor_in.unsqueeze(0).to(device)
    outputs = model(tensor_batch)
    attn_map = model.get_attention_maps(tensor_batch)[0, 0].cpu().numpy()
    preds = postprocess_batch(outputs, IMG_SIZE, conf_thresh=0.25, iou_thresh=0.45)[0]

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))

axes[0].imshow(orig_np)
axes[0].set_title("Input Fabric Sample")
axes[0].axis('off')

axes[1].imshow(orig_np)
axes[1].imshow(attn_map, cmap='jet', alpha=0.55)
axes[1].set_title("Attention-Guided Heatmap")
axes[1].axis('off')

axes[2].imshow(orig_np)
for box, score, label in zip(preds['boxes'], preds['scores'], preds['labels']):
    x1, y1, x2, y2 = box.tolist()
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='red', facecolor='none')
    axes[2].add_patch(rect)
    axes[2].text(x1, max(y1 - 5, 0), f"{CLASS_NAMES[int(label)]}: {score:.2f}",
                 color='white', fontsize=9, bbox=dict(facecolor='red', alpha=0.7, pad=1))
axes[2].set_title(f"AGL-YOLO Predictions ({len(preds['scores'])} detections)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

if gt_target.numel() > 0:
    print(f"[Ground truth boxes for this image] {gt_target.numpy()}")
else:
    print("[Ground truth] No defects annotated in this validation image.")